# CASPULE-NFSim Multiscale Coupling

_Investigation `caspule-nfsim-multi-scale-coupling` — coder reproduction notebook._

**Question.** Can bond-aware spatial MD (CASPULE) be tightly coupled to non-spatial rule-based kinetics (NFSim) via an observable detector, and does this coupling produce qualitatively different dynamics than running each simulator independently?

This workspace demonstrates a process-bigraph composite that couples CASPULE (bond-aware LAMMPS MD) to NFSim (BioNetGen rule-based kinetics) through a spatial observable detector. CASPULE evolves atoms and bond clusters; the detector identifies spatial patterns (dimers, trimers, polymers), optionally removes matched atoms from CASPULE, and injects them as molecule counts into NFSim's non-spatial pool. NFSim then evolves downstream kinetics. The composite provides a reusable scaffold for multiscale spatial-to-non-spatial coupling experiments.

---

**How to use this notebook**

1. **Run ▸ Run All Cells** to reproduce every study end to end.
2. To explore, edit a study's **Parameters** cell, then re-run from its **Run** section — the figures regenerate with your values.
3. The **Setup** cell at the top is collapsed scaffolding (environment + render helpers); expand it only to inspect. Set `RERUN = False` there to render the committed `runs.db` without re-simulating.

The prose states the **question and parameters**; the figures each run produces are the results.


In [ ]:
"""Setup — environment & helpers (runs automatically; open only to inspect).

Generated by vivarium-workbench (notebook_export). This cell resolves the repo,
registers the workspace Core, and defines the render helpers; the science lives
in the sections below. Each study is re-run live with the workspace's own
process-bigraph protocol and its figures rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-composite-nfsim-caspule/viva-composite-nfsim-caspule').is_dir():
    REPO = Path('/home/runner/work/viva-composite-nfsim-caspule/viva-composite-nfsim-caspule')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# Keep the output clean: composite discovery and the pydantic models emit
# benign UserWarnings (optional packages it skips, a field-name shadow) and the
# simulators emit DeprecationWarnings — none are actionable from a reproduction
# notebook, so silence those two categories. Remove this to see them.
import warnings as _warnings
_warnings.filterwarnings("ignore", category=UserWarning)
_warnings.filterwarnings("ignore", category=DeprecationWarning)

# --- standard process-bigraph protocol: register the workspace's Core ---
from pbg_composite_nfsim_caspule.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

def materialize_composite(spec_id, overrides=None):
    """Build a composite into an editable payload {name, description,
    parameters, state, _overrides, _id}. Works for @composite_generator
    functions via the process-bigraph registry (the common case); ``state``
    is the built document (process nodes at top level)."""
    import importlib
    from process_bigraph.composite_generator import (
        _REGISTRY, apply_core_extensions, build_generator)
    try:
        from process_bigraph.composite_generator import discover_generators
        discover_generators()
    except Exception:
        pass
    entry = _REGISTRY.get(spec_id)
    if entry is None:
        # editable installs are invisible to packages_distributions, so
        # discover alone can miss them — prime the module the id names.
        try:
            importlib.import_module(spec_id.rsplit('.', 1)[0])
            from process_bigraph.composite_generator import discover_generators as _dg
            _dg()
        except Exception:
            pass
        entry = _REGISTRY.get(spec_id)
    if entry is None:
        raise KeyError('composite not registered: ' + repr(spec_id))
    params = dict(entry.parameters or {})
    ov = {k: (v.get('default') if isinstance(v, dict) else None)
          for k, v in params.items()}
    if overrides:
        ov.update({k: val for k, val in overrides.items() if k in params})
    _c = apply_core_extensions(entry, core)
    try:
        doc = build_generator(entry, ov or None, core=_c)
    except TypeError:
        doc = entry.func(_c, **(ov or {}))
    return {'name': entry.name, 'description': entry.description,
            'parameters': params, 'state': doc, '_overrides': ov, '_id': spec_id}

import base64 as _b64, pathlib as _pl
_STATIC_VIZ_SCHEMES = ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:')
def _render_viz_process(address, config):
    """Render a Visualization process/step (e.g. ``local:<Class>``) to
    HTML: wrap it as a one-step composite and run it once — the viz writes
    Plotly HTML to its ``html`` output port. Mirrors the workbench's own
    env-worker viz path. Self-running vizes need no run data."""
    from process_bigraph import Composite
    _doc = {
        'inputs_store': {}, 'output_store': '',
        'visualization': {
            '_type': 'step', 'address': address,
            'config': {k: v for k, v in (config or {}).items()
                       if k not in ('inputs_map', 'sources')},
            'inputs': {}, 'outputs': {'html': ['output_store']},
        },
    }
    _comp = Composite({'state': _doc}, core=core)
    _comp.run(1)
    _html = _comp.state.get('output_store')
    if isinstance(_html, dict):
        _html = _html.get('value') or _html.get('_value') or ''
    return _html if isinstance(_html, str) else ''
def _render_one(address, config, runs_db, study_yaml, overrides=None):
    """Generic figure renderer (no workspace render_study_viz.py).
    A static ``image:``/``file:`` (or bare-path) address embeds the file;
    any other address (``local:<VizProcess>``) is run as a Visualization
    and its HTML returned. ``overrides`` (the composite's edited parameter
    dict) is merged into the viz config first so a self-running figure
    reflects edited parameters; a viz that does not accept those keys
    falls back to the base config."""
    addr = str(address or '')
    _head = addr.split('/', 1)[0]
    _is_static = (not addr) or (':' not in _head) or addr.startswith(_STATIC_VIZ_SCHEMES)
    if not _is_static:
        if core is None:
            return f'<p style="color:#b91c1c">no core to render {address}</p>'
        _base = {k: v for k, v in (config or {}).items()}
        _tries = ([{**_base, **overrides}] if overrides else []) + [_base]
        _err = ''
        for _cfg in _tries:
            try:
                _h = _render_viz_process(addr, _cfg)
            except Exception as _e:
                _err = (f'<p style="color:#b91c1c">visualization {address} not renderable yet: '
                        f'{type(_e).__name__}: {_e}</p>')
                continue
            return _h or f'<p style="color:#6b7280">{address} produced no HTML</p>'
        return _err
    for _scheme in _STATIC_VIZ_SCHEMES:
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: CASPULE-NFSim Dimerization (`caspule-nfsim-dimer`)

**Question.** How does spatial bond-formation in a CASPULE LJ monomer gas feed molecule counts into downstream NFSim rule-based kinetics, and what does the coupled temporal trajectory of dimer counts and thermo state look like?


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `pbg_composite_nfsim_caspule.composites.caspule-nfsim-dimer` | 0 | — |
| `baseline` | `pbg_composite_nfsim_caspule.composites.caspule-nfsim-dimer` | 0 | — |
| `baseline` | `pbg_composite_nfsim_caspule.composites.caspule-nfsim-dimer` | 30 | — |
| `baseline` | `pbg_composite_nfsim_caspule.composites.caspule-nfsim-dimer` | 30 | — |
| `baseline` | `pbg_composite_nfsim_caspule.composites.caspule-nfsim-dimer` | 30 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**Composite `pbg_composite_nfsim_caspule.composites.caspule-nfsim-dimer`** — `spec_pbg_composite_nfsim_caspule_composites_caspule_nfsim_dimer` (static spec file, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_pbg_composite_nfsim_caspule_composites_caspule_nfsim_dimer = load_spec(REPO / 'pbg_composite_nfsim_caspule/composites/caspule-nfsim-dimer.composite.yaml')
describe_spec(spec_pbg_composite_nfsim_caspule_composites_caspule_nfsim_dimer)

In [ ]:
# === Edit parameters for composite 'caspule-nfsim-dimer' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'caspule'  (local:CASPULEProcess)
spec_pbg_composite_nfsim_caspule_composites_caspule_nfsim_dimer['state']['caspule']['interval'] = 0.05
spec_pbg_composite_nfsim_caspule_composites_caspule_nfsim_dimer['state']['caspule']['config']['input_script'] = 'units lj\natom_style bond\nboundary p p p\nneighbor 0.5 bin\nneigh_modify every 1 delay 0 check yes\nregion box block 0 8 0 8 0 8\ncreate_box 1 box bond/types 1 extra/bond/per/atom 4 extra/special/per/atom 50\nmass 1 1.0\nlattice sc 1.6\ncreate_atoms 1 box\npair_style soft 2.0\npair_coeff * * 30.0\nbond_style harmonic\nbond_coeff 1 5.0 1.5\nminimize 1e-4 1e-4 100 100\nfix nve all nve/limit 0.05\nfix lan all langevin 0.4 0.4 1.0 12345\ntimestep 0.005\nfix mkbnd all bond/create 1 1 1 1.8 1 iparam 1 1 jparam 1 1\nthermo 5\n'
spec_pbg_composite_nfsim_caspule_composites_caspule_nfsim_dimer['state']['caspule']['config']['input_file'] = ''

# process 'nfsim'  (local:NFSimProcess)
spec_pbg_composite_nfsim_caspule_composites_caspule_nfsim_dimer['state']['nfsim']['interval'] = 0.5
spec_pbg_composite_nfsim_caspule_composites_caspule_nfsim_dimer['state']['nfsim']['config']['model_file'] = ''
spec_pbg_composite_nfsim_caspule_composites_caspule_nfsim_dimer['state']['nfsim']['config']['model_string'] = 'begin parameters\n    init_Dimer    0\n    init_Active   0\n    k_activate    0.1\n    k_deactivate  0.05\n    k_decay       0.01\nend parameters\nbegin molecule types\n    Dimer()\n    Active()\n    Decay()\nend molecule types\nbegin seed species\n    Dimer()    init_Dimer\n    Active()   init_Active\nend seed species\nbegin observables\n    Molecules Dimer    Dimer()\n    Molecules Active   Active()\nend observables\nbegin reaction rules\n    Dimer() -> Active()    k_activate\n    Active() -> Dimer()    k_deactivate\n    Active() -> Decay()    k_decay\nend reaction rules\n'
spec_pbg_composite_nfsim_caspule_composites_caspule_nfsim_dimer['state']['nfsim']['config']['n_steps'] = 20

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: caspule-nfsim-dimer ===
STUDY = 'caspule-nfsim-dimer'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Reproduce the recorded run(s). STEPS = composite steps.
STEPS_baseline = 100
INTERVAL_baseline = 1.0
STEPS_baseline = 100
INTERVAL_baseline = 1.0
STEPS_baseline = 30
INTERVAL_baseline = 1.0
STEPS_baseline = 30
INTERVAL_baseline = 1.0
STEPS_baseline = 30
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_composite_nfsim_caspule_composites_caspule_nfsim_dimer, {'interval': INTERVAL_baseline}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(pbg_composite_nfsim_caspule.composites.caspule-nfsim-dimer) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_composite_nfsim_caspule_composites_caspule_nfsim_dimer, {'interval': INTERVAL_baseline}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(pbg_composite_nfsim_caspule.composites.caspule-nfsim-dimer) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_composite_nfsim_caspule_composites_caspule_nfsim_dimer, {'interval': INTERVAL_baseline}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(pbg_composite_nfsim_caspule.composites.caspule-nfsim-dimer) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_composite_nfsim_caspule_composites_caspule_nfsim_dimer, {'interval': INTERVAL_baseline}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(pbg_composite_nfsim_caspule.composites.caspule-nfsim-dimer) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_composite_nfsim_caspule_composites_caspule_nfsim_dimer, {'interval': INTERVAL_baseline}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(pbg_composite_nfsim_caspule.composites.caspule-nfsim-dimer) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**caspule-nfsim-dimer-world**


In [ ]:
# caspule-nfsim-dimer-world
show_viz(_render_one('local:TimeSeriesFromObservables', {'observables': ['num_atoms', 'temperature'], 'title': 'CASPULE monomer pool — dimerization depletion'}, RUNS_DB, STUDY_YAML))

**caspule-nfsim-dimer-mechanism**


In [ ]:
# caspule-nfsim-dimer-mechanism
show_viz(_render_one('local:NFSimCaspuleMechanism', {'observables': ['num_clusters', 'species.Dimer', 'species.Active'], 'title': 'CASPULE clusters + NFSim species — dimerization mechanism'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| caspule-nfsim-dimer-behaves _(primary)_ | kind=derived_scalar field=num_bonds | op > value 0 |


## Study: CASPULE-NFSim Polymer Fractionation (`caspule-nfsim-polymer`)

**Question.** How does size-based fractionation of spatial bond clusters (dimers, trimers, and larger polymers) into distinct NFSim molecule species shape the downstream non-spatial activation and decay kinetics?


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `pbg_composite_nfsim_caspule.composites.caspule-nfsim-polymer` | 30 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**Composite `pbg_composite_nfsim_caspule.composites.caspule-nfsim-polymer`** — `spec_pbg_composite_nfsim_caspule_composites_caspule_nfsim_polymer` (static spec file, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_pbg_composite_nfsim_caspule_composites_caspule_nfsim_polymer = load_spec(REPO / 'pbg_composite_nfsim_caspule/composites/caspule-nfsim-polymer.composite.yaml')
describe_spec(spec_pbg_composite_nfsim_caspule_composites_caspule_nfsim_polymer)

In [ ]:
# === Edit parameters for composite 'caspule-nfsim-polymer' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'caspule'  (local:CASPULEProcess)
spec_pbg_composite_nfsim_caspule_composites_caspule_nfsim_polymer['state']['caspule']['interval'] = 0.05
spec_pbg_composite_nfsim_caspule_composites_caspule_nfsim_polymer['state']['caspule']['config']['input_script'] = 'units lj\natom_style bond\nboundary p p p\nneighbor 0.5 bin\nneigh_modify every 1 delay 0 check yes\nregion box block 0 6 0 6 0 6\ncreate_box 1 box bond/types 1 extra/bond/per/atom 4 extra/special/per/atom 50\nmass 1 1.0\nlattice sc 1.6\ncreate_atoms 1 box\npair_style soft 2.0\npair_coeff * * 30.0\nbond_style harmonic\nbond_coeff 1 5.0 1.5\nminimize 1e-4 1e-4 100 100\nfix nve all nve/limit 0.05\nfix lan all langevin 0.4 0.4 1.0 12345\ntimestep 0.005\nfix mkbnd all bond/create 10 1 1 1.8 1 iparam 4 1 jparam 4 1\nthermo 5\n'
spec_pbg_composite_nfsim_caspule_composites_caspule_nfsim_polymer['state']['caspule']['config']['input_file'] = ''

# process 'nfsim'  (local:NFSimProcess)
spec_pbg_composite_nfsim_caspule_composites_caspule_nfsim_polymer['state']['nfsim']['interval'] = 0.5
spec_pbg_composite_nfsim_caspule_composites_caspule_nfsim_polymer['state']['nfsim']['config']['model_file'] = ''
spec_pbg_composite_nfsim_caspule_composites_caspule_nfsim_polymer['state']['nfsim']['config']['model_string'] = 'begin parameters\n    init_Dimer    0\n    init_Trimer   0\n    init_Polymer  0\n    init_Active   0\n    k_dimer_act   0.30\n    k_trimer_act  0.05\n    k_polymer_dec 0.02\n    k_active_dec  0.05\nend parameters\nbegin molecule types\n    Dimer()\n    Trimer()\n    Polymer()\n    Active()\n    Decay()\nend molecule types\nbegin seed species\n    Dimer()    init_Dimer\n    Trimer()   init_Trimer\n    Polymer()  init_Polymer\n    Active()   init_Active\nend seed species\nbegin observables\n    Molecules Dimer    Dimer()\n    Molecules Trimer   Trimer()\n    Molecules Polymer  Polymer()\n    Molecules Active   Active()\nend observables\nbegin reaction rules\n    Dimer()   -> Active()    k_dimer_act\n    Trimer()  -> Active()    k_trimer_act\n    Active()  -> Decay()     k_active_dec\n    Polymer() -> Decay()     k_polymer_dec\nend reaction rules\n'
spec_pbg_composite_nfsim_caspule_composites_caspule_nfsim_polymer['state']['nfsim']['config']['n_steps'] = 20

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: caspule-nfsim-polymer ===
STUDY = 'caspule-nfsim-polymer'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Reproduce the recorded run(s). STEPS = composite steps.
STEPS_baseline = 30
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_composite_nfsim_caspule_composites_caspule_nfsim_polymer, {'interval': INTERVAL_baseline}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(pbg_composite_nfsim_caspule.composites.caspule-nfsim-polymer) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**caspule-nfsim-polymer-world**


In [ ]:
# caspule-nfsim-polymer-world
show_viz(_render_one('local:TimeSeriesFromObservables', {'observables': ['num_atoms', 'largest_cluster', 'num_bonds'], 'title': 'CASPULE polymer growth — atom and cluster trajectory'}, RUNS_DB, STUDY_YAML))

**caspule-nfsim-polymer-mechanism**


In [ ]:
# caspule-nfsim-polymer-mechanism
show_viz(_render_one('local:NFSimCaspuleMechanism', {'observables': ['num_clusters', 'largest_cluster', 'species.Dimer', 'species.Trimer', 'species.Active'], 'title': 'CASPULE clusters + NFSim species — polymer fractionation mechanism'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| caspule-nfsim-polymer-behaves _(primary)_ | kind=derived_scalar field=largest_cluster | op > value 0 |
